# Adaptive Traffic Signal Control Using Computer Vision

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SandeshBansode/Adaptive-Traffic-Signal-Control-Using-Computer-Vision/blob/main/colab_demo.ipynb)

Four cameras (North, East, South, West) count the vehicles waiting at a junction with YOLOv4-tiny.
The adaptive controller uses those queues to run the traffic lights:

- **Paired phases:** North + South go green together, then East + West.
- **Green length** follows the longest queue.
- **Change intervals:** yellow and all-red times come from the ITE formulas.
- **Paused video on red:** a red approach's video pauses, so its traffic waits like real cars.
- **Counting area:** only vehicles between the drawn road lines are counted, and parked cars are ignored.

**To run everything:** choose *Runtime → Run all*. It takes about 5 minutes on the free CPU runtime.

## 1. Get the code and install

In [ ]:
import os

REPO_URL = "https://github.com/SandeshBansode/Adaptive-Traffic-Signal-Control-Using-Computer-Vision.git"
BRANCH = "main"

if not os.path.exists("detect_and_signal.py"):
    if not os.path.exists("traffic"):
        !git clone --depth 1 -b {BRANCH} {REPO_URL} traffic
    %cd traffic

!pip install -q "opencv-python-headless>=4.5.4,<5" eclipse-sumo traci pytest

## 2. Run the tests
There are 53 unit tests. They cover the controller (phases, timing rules, gap-out, starvation and emergency preemption), the road lines, the parked-car filter and the tracker.

In [ ]:
!python -m pytest -q

## 3. Check the road lines
Only vehicles **between the two lines** are counted. The lines come from `road_roi.json`.
**Green boxes** are vehicles that will be counted. **Grey boxes** are outside the lines and are ignored.

In [ ]:
import os, cv2, matplotlib.pyplot as plt
import detect_and_signal as ds, ui
from road_roi import RoadROI, load_config, ground_point

class_names = ds.load_class_names()
net, output_layers = ds.load_network()
config = load_config()

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, label, name in zip(axes.flat, ds.DEFAULT_LABELS, ds.DEFAULT_VIDEOS):
    cap = cv2.VideoCapture(os.path.join(ds.VIDEOS_DIR, name))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        ax.set_title(f"{label}: {name} not found"); ax.axis("off"); continue
    h, w = frame.shape[:2]
    roi = RoadROI.from_config(name, config)
    if roi is not None:
        ui.road_zone(frame, roi, "green")
    inside = 0
    for rect in ds.detect_vehicles(net, output_layers, class_names, frame):
        sx, sy, ex, ey = map(int, rect)
        counted = roi is None or roi.contains(ground_point(rect), w, h)
        inside += counted
        cv2.rectangle(frame, (sx, sy), (ex, ey), (60, 220, 60) if counted else (150, 150, 150), 2)
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{label} ({name}): {inside} counted" + ("" if roi else "  - NO LINES, whole frame"))
    ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Live run: adaptive control
This runs the full pipeline without windows (`--headless`), using the four videos in `videos/`.
When it finishes it saves:
- the recorded 2×2 camera grid
- a summary image
- `outputs/latest_run.json`

In [ ]:
import shutil, json
from IPython.display import Video, Image, display

def run_mode(mode, extra=""):
    !python detect_and_signal.py --headless --mode {mode} {extra}
    shutil.copy("outputs/latest_run.json", f"outputs/run_{mode}.json")
    shutil.copy("outputs/summary.png", f"outputs/summary_{mode}.png")
    # Re-encode to H.264 so the browser can play it (OpenCV writes mp4v)
    !ffmpeg -y -loglevel error -i outputs/traffic_detection_output.mp4 -vf scale=960:-2 -c:v libx264 -crf 30 -preset veryfast -pix_fmt yuv420p outputs/{mode}_h264.mp4

run_mode("adaptive")
display(Video("outputs/adaptive_h264.mp4", embed=True, width=960))
display(Image("outputs/summary_adaptive.png"))

## 5. Same videos with a fixed 30 s timer, for comparison
The fixed timer gives each pair a 30 s green whether or not cars are waiting.

In [ ]:
run_mode("fixed")
display(Image("outputs/summary_fixed.png"))

import pandas as pd
rows = []
for mode in ("adaptive", "fixed"):
    r = json.load(open(f"outputs/run_{mode}.json"))
    waits = [l["avg_wait"] for l in r["lanes"] if l["avg_wait"] is not None]
    rows.append({"mode": mode,
                 "time to clear all traffic (s)": r["sim_time"],
                 "greens": len(r["phases"]),
                 "avg red wait per approach (s)": round(sum(waits) / len(waits), 1) if waits else None,
                 "vehicles counted": sum(l["counted"] for l in r["lanes"])})
pd.DataFrame(rows).set_index("mode")

## 6. Queue simulation: 30 minutes of random traffic
Vehicles arrive at random (a Poisson process), in the proportions the cameras counted. The same arrivals go through both controllers.

In [ ]:
!python simulate_comparison.py
display(Image("outputs/comparison_chart.png"))

## 7. SUMO microsimulation check
[SUMO](https://eclipse.dev/sumo/) simulates every car's driving at a 4-way junction.
The script runs it once with the fixed timer and once with the adaptive controller, over the same 702 trips.

In [ ]:
!python sumo_sim/traci_control.py --mode both

## 8. (Optional) Use your own videos
Upload four straight-road videos in the order **North, East, South, West**. They replace `videos/1.mp4` to `videos/4.mp4`.

Next, set the road lines in the cell after this one, then run sections 3 and 4 again.

In [ ]:
from google.colab import files
for i, label in enumerate(ds.DEFAULT_LABELS, start=1):
    print(f"Upload the {label} video")
    uploaded = files.upload()
    if uploaded:
        name = next(iter(uploaded))
        os.replace(name, f"videos/{i}.mp4")
        print(f"  saved as videos/{i}.mp4")

In [ ]:
# Road lines, as fractions of the frame: x from 0 (left) to 1 (right), y from 0 (top) to 1 (bottom).
# Each line is two points along one edge of the lanes heading INTO the junction.
# Check the result with section 3. On a desktop, `python calibrate_roi.py` lets you click the lines instead.
from road_roi import save_roi

LINES = {
    "1.mp4": {"left": [[0.34, 0.10], [0.01, 0.63]], "right": [[0.72, 0.10], [0.99, 0.58]]},
    "2.mp4": {"left": [[0.34, 0.10], [0.01, 0.63]], "right": [[0.72, 0.10], [0.99, 0.58]]},
    "3.mp4": {"left": [[0.34, 0.10], [0.01, 0.63]], "right": [[0.72, 0.10], [0.99, 0.58]]},
    "4.mp4": {"left": [[0.34, 0.10], [0.01, 0.63]], "right": [[0.72, 0.10], [0.99, 0.58]]},
}
for name, lines in LINES.items():
    save_roi(name, RoadROI(lines["left"], lines["right"]))
print("Saved road_roi.json - now re-run sections 3 and 4.")

## 9. (Optional) Dashboard
This starts the Streamlit dashboard and opens it in a new browser tab. Run sections 4 to 7 first, so it has results to show.

In [ ]:
!pip install -q "streamlit>=1.50" altair
import subprocess
subprocess.Popen(["streamlit", "run", "dashboard.py", "--server.headless", "true", "--server.port", "8501"])
import time; time.sleep(8)
from google.colab import output
output.serve_kernel_port_as_window(8501)